# Demo: Whisper grande + reglas + LLM (sin GPU)

El análisis de resultados mostró que **whisper-large-v3-turbo + las reglas del paper** (31 % de WER en test) rinde
mejor que whisper-small con fine-tuning (D, 37 %). Este notebook:
1. **Mide** el pipeline completo *turbo + reglas + LLM* sobre los mismos hablantes de test, usando las transcripciones
   de turbo que el notebook principal ya guardó en Drive. Así la demo se apoya en un número medido.
2. **Elige audios de demo** donde turbo solo falla y el pipeline acierta.
3. **Demo en vivo con Gradio:** la transcripción la hace **Groq** con `whisper-large-v3-turbo` en sus servidores,
   así que **no hace falta GPU**. Usá un entorno de CPU.

Secretos (ícono de llave, con "Acceso al notebook"): `GROQ_API_KEY` (transcripción) y la clave del LLM
(`OPENAI_API_KEY` por defecto).

**Aclaración honesta:** el turbo de la API de Groq es el mismo modelo que se evaluó, pero Groq puede procesar el audio
de forma un poco distinta. Los números de la sección 2 corresponden al turbo corrido en Colab.

In [ ]:
%pip install -q "jiwer==4.0.0" num2words openai anthropic "gradio>=4.44" "huggingface-hub>=0.34,<1.0"

## Configuración

In [ ]:
import os, sys, json, re, time, random, zipfile, unicodedata, glob, hashlib, shutil
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

CFG = dict(
    nbest=5,
    llm_proveedor="openai_compatible",   # OpenAI por defecto (rápido)
    llm_modelo="gpt-4.1-mini",
    llm_esfuerzo="low",
    llm_base_url=None,                   # None = OpenAI
    llm_hilos=8,
    llm_ejemplos=3,
    seed=42,
    asr_modelo="whisper-large-v3-turbo",             # transcripción por API de Groq
    asr_base_url="https://api.groq.com/openai/v1",
)
# LLM por Groq en vez de OpenAI: descomentá
# CFG.update(llm_base_url="https://api.groq.com/openai/v1", llm_modelo="openai/gpt-oss-120b", llm_hilos=2)

if "google.colab" in sys.modules and not os.environ.get("SDIVERSE_BASE"):
    from google.colab import drive
    drive.mount("/content/drive")
BASE = os.environ.get("SDIVERSE_BASE", "/content/drive/MyDrive/Shippear/sdiverse_hackathon")
RES = os.path.join(BASE, "resultados")
DEMO_DIR = os.path.join(BASE, "demo_turbo")
os.makedirs(DEMO_DIR, exist_ok=True)
ZIP_DIR = os.path.dirname(BASE)

## 1. Datos
Las transcripciones de val/test del principal (incluye turbo y D), el TSV y las hipótesis de whisper-small en
entrenamiento (para los ejemplos que recibe el LLM).

In [ ]:
for p in (os.path.join(BASE, "splits.csv"), os.path.join(RES, "pred_A.csv"),
          os.path.join(RES, "transcripciones_val_test.csv")):
    if not os.path.exists(p):
        raise FileNotFoundError(f"Falta {p}: corré antes el notebook principal.")

ZIP = None
for z in sorted(glob.glob(os.path.join(ZIP_DIR, "*.zip"))):
    try:
        if any(n.endswith("s-diverse.tsv") for n in zipfile.ZipFile(z).namelist()):
            ZIP = zipfile.ZipFile(z); break
    except zipfile.BadZipFile:
        pass
assert ZIP is not None, f"No encuentro el zip con s-diverse.tsv en {ZIP_DIR}"
tsv = pd.read_csv(ZIP.open([n for n in ZIP.namelist() if n.endswith("s-diverse.tsv")][0]), sep="\t")

splits = pd.read_csv(os.path.join(BASE, "splits.csv"))
predA = pd.read_csv(os.path.join(RES, "pred_A.csv"), keep_default_na=False)
predA["nbest"] = predA.nbest.map(json.loads)
df = (tsv.merge(splits[["Sample_ID", "split"]], on="Sample_ID")
         .merge(predA.rename(columns={"hyp": "A", "nbest": "A_nbest"}), on="Sample_ID"))
EV = pd.read_csv(os.path.join(RES, "transcripciones_val_test.csv"), keep_default_na=False)
EV = EV.rename(columns={"Ref. large-v3-turbo": "Turbo"})
assert "Turbo" in EV, "El principal no guardó la transcripción de turbo (CFG['evaluar_referencia'])."
print(f"val+test: {len(EV)} segmentos | test: {(EV.split == 'test').sum()}")

## Funciones compartidas con el notebook principal (normalización, WER, reglas)

In [ ]:
import jiwer
from num2words import num2words

def _num_a_palabras(m):
    s = m.group(0).replace(".", "")
    try:
        return " " + num2words(int(s), lang="es") + " "
    except Exception:
        return " " + s + " "

def normalizar(texto, quitar_tildes=False):
    t = unicodedata.normalize("NFC", str(texto)).lower()
    t = t.replace("<unk>", " ")
    t = re.sub(r"\d{1,3}(?:\.\d{3})+(?!\d)|\d+", _num_a_palabras, t)
    t = re.sub(r"[^\w\s]|_", " ", t)
    if quitar_tildes:
        t = unicodedata.normalize("NFD", t)
        t = re.sub("[̀́̈]", "", t)   # grave, agudo, diéresis (la virgulilla de la ñ queda)
        t = unicodedata.normalize("NFC", t)
    return " ".join(t.split())

def contar_errores(ref, hyp):
    """Errores de un segmento con jiwer. ref/hyp ya normalizados."""
    r, h = ref.split(), hyp.split()
    if not r:
        return dict(S=0, D=0, I=len(h), N=0)
    if not h:
        return dict(S=0, D=len(r), I=0, N=len(r))
    o = jiwer.process_words(ref, hyp)
    return dict(S=o.substitutions, D=o.deletions, I=o.insertions, N=o.hits + o.substitutions + o.deletions)

def wer_segmentos(tabla, col_hyp, sin_tildes=False):
    """Una fila por segmento con S, D, I, N y WER."""
    filas = []
    for ref, hyp in zip(tabla.Transcription, tabla[col_hyp].fillna("")):
        filas.append(contar_errores(normalizar(ref, sin_tildes), normalizar(hyp, sin_tildes)))
    e = pd.DataFrame(filas, index=tabla.index, columns=["S", "D", "I", "N"])   # columnas fijas: tabla puede venir vacía
    e["WER"] = (e.S + e.D + e.I) / e.N.replace(0, np.nan)
    return e

def resumen_wer(tabla, metodos, por=None, sin_tildes=False):
    """WER agregado (suma de errores / suma de palabras) por grupo, con conteos."""
    tabla = tabla[tabla.Transcription.map(normalizar).str.len() > 0]
    claves = [por] if por else []
    base = tabla.groupby(claves) if por else tabla.assign(_g="TOTAL").groupby("_g")
    out = base.agg(segmentos=("Sample_ID", "size"), hablantes=("Speaker_ID", "nunique"))
    out["palabras"] = tabla.Transcription.map(lambda x: len(normalizar(x).split())).groupby(
        tabla[por] if por else pd.Series("TOTAL", index=tabla.index)).sum()
    for m in metodos:
        if m not in tabla or tabla[m].isna().all():
            continue
        e = wer_segmentos(tabla, m, sin_tildes)
        g = e.groupby(tabla[por] if por else pd.Series("TOTAL", index=tabla.index))
        s = g.sum()
        out[m] = (100 * (s.S + s.D + s.I) / s.N).round(1)
    return out

def _reducir_palabra(w):
    if len(w) <= 15:
        return w
    for L in range(1, len(w) // 2 + 1):
        if (w[:L] * (len(w) // L + 1))[:len(w)] == w:
            return w[:L]
    return w

def _clave(w):
    return re.sub(r"[^\w]", "", w.lower())

def postprocesar_reglas(texto, max_n=40):
    """Las 3 reglas del paper (sección 3.4). Solo borran: nunca agregan palabras."""
    toks = [_reducir_palabra(w) for w in str(texto).split()]
    out = []
    for w in toks:                                         # (ii) palabra repetida
        if out and _clave(w) and _clave(w) == _clave(out[-1]):
            continue
        out.append(w)
    toks, claves = out, [_clave(w) for w in out]
    cambio = True
    while cambio:                                          # (iii) frase repetida
        cambio = False
        for n in range(min(len(toks) // 2, max_n), 1, -1):
            i = 0
            while i + 2 * n <= len(toks):
                if claves[i:i + n] == claves[i + n:i + 2 * n]:
                    del toks[i + n:i + 2 * n]; del claves[i + n:i + 2 * n]; cambio = True
                else:
                    i += 1
    return " ".join(toks)

# Pruebas rápidas (si alguna falla, el notebook se detiene acá)
assert normalizar("¿Esto tiene tratamiento?\" y dice: \"no, no hay\".") == "esto tiene tratamiento y dice no no hay"
assert normalizar("Eso fue en el 20 y en 2009 <unk>") == "eso fue en el veinte y en dos mil nueve"
assert normalizar("Está aquí, señor", quitar_tildes=True) == "esta aqui señor"
assert postprocesar_reglas("y entonces y entonces y entonces me fui me fui") == "y entonces me fui"
assert postprocesar_reglas("jajajajajajajajajaja vale") == "ja vale"
assert contar_errores("a b c", "a x c d") == dict(S=1, D=0, I=1, N=3)
print("normalización, WER y reglas: OK")

## Funciones de LLM (idénticas al principal)

In [ ]:
SYSTEM_LLM = """Corregís transcripciones automáticas (ASR) de habla espontánea en español producida por personas con disartria (ELA, Parkinson o secuelas de ACV). Recibís una o varias hipótesis del reconocedor para el MISMO audio, ordenadas de más a menos probable.

Devolvé la transcripción más probable de lo que la persona dijo.
- Corregí solo errores típicos de reconocimiento: palabras que suenan parecido, palabras inexistentes, cortes o uniones de palabras, repeticiones en bucle del reconocedor.
- Usá las hipótesis alternativas para decidir entre palabras dudosas.
- No agregues información que no esté en ninguna hipótesis, no resumas, no parafrasees y no corrijas la gramática ni el estilo de la persona.
- Mantené muletillas y repeticiones naturales ("eh", "o sea", "no, no").
- Si la primera hipótesis parece correcta, devolvela sin cambios. Ante la duda, cambiá lo menos posible.
- Respondé solo con la transcripción final, sin comillas ni comentarios."""

def obtener_secreto(nombre):
    if os.environ.get(nombre):
        return os.environ[nombre]
    try:
        from google.colab import userdata
        return userdata.get(nombre)
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(nombre)
    except Exception:
        return None

_cliente = {}
def _llm_anthropic(mensajes):
    import anthropic
    if "a" not in _cliente:
        _cliente["a"] = anthropic.Anthropic(api_key=obtener_secreto("ANTHROPIC_API_KEY"), max_retries=4)
    kw = dict(model=CFG["llm_modelo"], max_tokens=4000, system=SYSTEM_LLM, messages=mensajes)
    if CFG["llm_esfuerzo"]:
        kw["output_config"] = {"effort": CFG["llm_esfuerzo"]}
    if CFG["llm_modelo"] in ("claude-opus-5-5", "claude-opus-5", "claude-fable-5-1", "claude-sonnet-5-5"):
        r = _cliente["a"].beta.messages.create(betas=["server-side-fallback-2026-07-01"], fallbacks="default", **kw)
    else:
        r = _cliente["a"].messages.create(**kw)
    if r.stop_reason == "refusal":
        return None
    return "".join(b.text for b in r.content if b.type == "text")

def nombre_clave():
    if CFG["llm_proveedor"] == "anthropic":
        return "ANTHROPIC_API_KEY"
    return "GROQ_API_KEY" if "groq" in (CFG["llm_base_url"] or "") else "OPENAI_API_KEY"

def _llm_openai(mensajes):
    from openai import OpenAI
    if "o" not in _cliente:
        _cliente["o"] = OpenAI(api_key=obtener_secreto(nombre_clave()) or "sin-clave",
                               base_url=CFG["llm_base_url"], max_retries=0)
    r = _cliente["o"].chat.completions.create(
        model=CFG["llm_modelo"], messages=[{"role": "system", "content": SYSTEM_LLM}] + mensajes)
    texto = r.choices[0].message.content or ""
    return re.sub(r"<think>.*?</think>", "", texto, flags=re.S)   # modelos que razonan en voz alta

ERRORES_DEFINITIVOS = {"NotFoundError", "AuthenticationError", "PermissionDeniedError", "BadRequestError",
                       "UnprocessableEntityError"}

def probar_llm():
    """Una llamada de prueba antes de lanzar todas. Devuelve True si el proveedor y el modelo responden."""
    f = {"anthropic": _llm_anthropic, "openai_compatible": _llm_openai}[CFG["llm_proveedor"]]
    try:
        f([{"role": "user", "content": "Hipótesis 1: hola buenos días"}])
        print(f"LLM OK: {CFG['llm_proveedor']} / {CFG['llm_modelo']}")
        return True
    except Exception as e:
        print(f"El LLM no responde ({type(e).__name__}): {str(e)[:300]}")
        if type(e).__name__ == "NotFoundError":
            print(f"El modelo '{CFG['llm_modelo']}' no existe en el proveedor. Modelos disponibles:")
            try:
                from openai import OpenAI
                if CFG["llm_proveedor"] == "openai_compatible":
                    cli = OpenAI(api_key=obtener_secreto(nombre_clave()), base_url=CFG["llm_base_url"])
                    print(sorted(m.id for m in cli.models.list().data))
            except Exception:
                pass
            print('Elegí uno grande de propósito general y poné CFG["llm_modelo"] = "..." en la configuración.')
        elif type(e).__name__ in ("AuthenticationError", "PermissionDeniedError"):
            print(f"Revisá el secreto {nombre_clave()}: clave inválida o sin 'Acceso al notebook'.")
        return False

RUTA_CACHE = os.path.join(RES, "cache_llm.json")
CACHE_LLM = json.load(open(RUTA_CACHE, encoding="utf-8")) if os.path.exists(RUTA_CACHE) else {}

def llamar_llm(mensajes):
    clave = hashlib.sha1(json.dumps([CFG["llm_proveedor"], CFG["llm_modelo"], SYSTEM_LLM, mensajes],
                                    ensure_ascii=False).encode()).hexdigest()
    if clave in CACHE_LLM:
        return CACHE_LLM[clave]
    f = {"anthropic": _llm_anthropic, "openai_compatible": _llm_openai}[CFG["llm_proveedor"]]
    for intento in range(6):   # reintentos con espera creciente: los planes gratuitos limitan llamadas por minuto
        try:
            salida = f(mensajes)
            break
        except Exception as e:
            # Modelo inexistente, clave inválida o pedido mal formado: reintentar no sirve
            if intento == 5 or type(e).__name__ in ERRORES_DEFINITIVOS:
                raise
            espera = min(60, 5 * 2 ** intento)
            print(f"LLM: {type(e).__name__}, reintento en {espera}s")
            time.sleep(espera)
    CACHE_LLM[clave] = salida
    return salida

def formatear_hipotesis(hyps):
    return "\n".join(f"Hipótesis {i+1}: {h}" for i, h in enumerate(hyps))

def hipotesis_para_llm(greedy, nbest):
    vistas, hyps = set(), []
    for h in [greedy] + list(nbest):
        h = postprocesar_reglas(h)
        if h.strip() and normalizar(h) not in vistas:
            vistas.add(normalizar(h)); hyps.append(h)
    return hyps[:CFG["nbest"]] or [greedy]

def elegir_ejemplos(k):
    """Ejemplos few-shot de hablantes de ENTRENAMIENTO, uno por enfermedad, con errores moderados."""
    tr = df[(df.split == "train") & (df.Duration.between(5, 20))].copy()
    tr["wer"] = wer_segmentos(tr, "A").WER
    tr = tr[tr.wer.between(0.15, 0.6)].sort_values("wer")
    ej = tr.groupby("Disease").head(1).head(k)
    msgs = []
    for _, r in ej.iterrows():
        msgs += [{"role": "user", "content": formatear_hipotesis(hipotesis_para_llm(r.A, r.A_nbest))},
                 {"role": "assistant", "content": r.Transcription.replace("<unk>", "").strip()}]
    return msgs, ej.Sample_ID.tolist()

N_DESCARTADAS = {"n": 0}
def corregir(greedy, nbest, ejemplos):
    hyps = hipotesis_para_llm(greedy, nbest)
    base = hyps[0]
    if not base.strip():
        return base
    try:
        salida = llamar_llm(ejemplos + [{"role": "user", "content": formatear_hipotesis(hyps)}])
    except Exception as e:
        N_DESCARTADAS["fallos"] = N_DESCARTADAS.get("fallos", 0) + 1
        print("Error de LLM:", type(e).__name__, str(e)[:200])
        return base
    salida = (salida or "").strip().strip('"“”«»').strip()
    nb, ns = len(base.split()), len(salida.split())
    if not salida or ns < 0.5 * nb or ns > 1.5 * nb + 3:
        N_DESCARTADAS["n"] += 1
        return base
    return salida

def aplicar_llm(tabla, col_greedy, col_nbest, ejemplos):
    from concurrent.futures import ThreadPoolExecutor
    filas = list(zip(tabla[col_greedy], tabla[col_nbest]))
    N_DESCARTADAS["fallos"] = 0
    with ThreadPoolExecutor(CFG["llm_hilos"]) as ex:
        out = list(tqdm(ex.map(lambda f: corregir(f[0], f[1], ejemplos), filas), total=len(filas), desc="LLM"))
    json.dump(CACHE_LLM, open(RUTA_CACHE, "w", encoding="utf-8"), ensure_ascii=False)
    if N_DESCARTADAS["fallos"]:
        print(f"ATENCIÓN: {N_DESCARTADAS['fallos']} de {len(filas)} llamadas fallaron y quedaron SIN corregir. "
              "Volvé a correr la celda: las exitosas salen de la caché y solo se reintentan las fallidas.")
    return out

RUTA_CACHE = os.path.join(RES, "cache_llm_demo.json")
CACHE_LLM = json.load(open(RUTA_CACHE, encoding="utf-8")) if os.path.exists(RUTA_CACHE) else {}
if not obtener_secreto(nombre_clave()):
    raise RuntimeError(f"Falta el secreto {nombre_clave()} (ícono de llave en Colab).")
if not os.environ.get("SDIVERSE_PRUEBA_LOCAL") and not probar_llm():
    raise RuntimeError("El LLM no responde: corregí CFG (ver el mensaje de arriba) y volvé a correr desde acá.")

## 2. Medir turbo + reglas + LLM

Mismo procedimiento que B y D: reglas primero, después el LLM con los 3 ejemplos de entrenamiento y la salvaguarda de
largo. Turbo se transcribió con una sola hipótesis, así que el LLM recibe una sola.

Para comparar con D en igualdad de condiciones, uso el **D del notebook de contexto si existe** (mismo LLM,
`gpt-4.1-mini`); si no, el del principal.

In [ ]:
def bootstrap_hablantes(tabla, m1, m2, n=5000, seed=0):
    e1, e2 = wer_segmentos(tabla, m1), wer_segmentos(tabla, m2)
    g = pd.DataFrame({"spk": tabla.Speaker_ID, "a": e1.S + e1.D + e1.I, "b": e2.S + e2.D + e2.I,
                      "N": e1.N}).groupby("spk").sum()
    rng = np.random.default_rng(seed)
    d = [100 * (s.a.sum() - s.b.sum()) / s.N.sum() for s in (g.iloc[rng.integers(0, len(g), len(g))] for _ in range(n))]
    return np.percentile(d, [2.5, 50, 97.5])

EJEMPLOS, ids_ej = elegir_ejemplos(CFG["llm_ejemplos"])
assert not set(ids_ej) & set(EV.Sample_ID)
EV["Turbo+PP"] = EV["Turbo"].map(postprocesar_reglas)
EV["_sin_nbest"] = [[] for _ in range(len(EV))]
EV["Turbo+PP+LLM"] = aplicar_llm(EV, "Turbo", "_sin_nbest", EJEMPLOS)

ruta_ctx = os.path.join(RES, "transcripciones_contexto.csv")
if os.path.exists(ruta_ctx):
    dc = pd.read_csv(ruta_ctx, keep_default_na=False)[["Sample_ID", "D"]].rename(columns={"D": "D (mismo LLM)"})
    EV = EV.merge(dc, on="Sample_ID", how="left")
METODOS = [m for m in ["A", "A+PP", "D", "D (mismo LLM)", "Turbo", "Turbo+PP", "Turbo+PP+LLM"] if m in EV]

VAL, TEST = EV[EV.split == "val"], EV[EV.split == "test"]
print("Validación:"); display(resumen_wer(VAL, [m for m in METODOS if m != "D (mismo LLM)"]))
for nombre, por in [("total", None), ("por enfermedad", "Disease"), ("por hablante", "Speaker_ID")]:
    print(f"\n=== WER (%) en TEST, {nombre} ===")
    display(resumen_wer(TEST, METODOS, por))
lo, med, hi = bootstrap_hablantes(TEST, "Turbo+PP", "Turbo+PP+LLM")
print(f"\nEl LLM sobre turbo+reglas mejora {med:.1f} puntos (IC 95% por hablantes: {lo:.1f} a {hi:.1f})")

# El LLM se usa en la demo solo si mejora en VALIDACIÓN (no miro test para decidir)
w = resumen_wer(VAL, ["Turbo+PP", "Turbo+PP+LLM"]).iloc[0]
USAR_LLM = float(w["Turbo+PP+LLM"]) <= float(w["Turbo+PP"])
FINAL = "Turbo+PP+LLM" if USAR_LLM else "Turbo+PP"
print(f"Pipeline de la demo: {FINAL}")
EV.drop(columns=["_sin_nbest"]).to_csv(os.path.join(RES, "transcripciones_turbo_llm.csv"), index=False)

## 3. Audios para la demo
Segmentos de **test** donde turbo solo falla y el pipeline final acierta (≤ 35 % de WER), de hasta 30 s, uno por
hablante cuando se puede. Se extraen del zip a `demo_turbo/`.

In [ ]:
from IPython.display import Audio
t = TEST.copy()
t["WER_turbo"] = wer_segmentos(t, "Turbo").WER
t["WER_final"] = wer_segmentos(t, FINAL).WER
t = t[(t.Duration <= 30) & (t.WER_final <= 0.35)].assign(ganancia=lambda x: x.WER_turbo - x.WER_final)
t = t.sort_values("ganancia", ascending=False)
demo = t.groupby("Speaker_ID").head(1).head(5)
if len(demo) < 5:
    demo = pd.concat([demo, t[~t.Sample_ID.isin(demo.Sample_ID)].head(5 - len(demo))])
nombres = {os.path.basename(n)[:-4]: n for n in ZIP.namelist() if n.endswith(".wav")}
for _, r in demo.iterrows():
    destino = os.path.join(DEMO_DIR, f"{r.Sample_ID}.wav")
    with ZIP.open(nombres[r.Sample_ID]) as src, open(destino, "wb") as dst:
        shutil.copyfileobj(src, dst)
    print(f"\n[{r.Sample_ID} | {r.Disease} | {r.Intelligibility}] WER turbo {100*r.WER_turbo:.0f}% → {100*r.WER_final:.0f}%")
    print("  Referencia :", r.Transcription)
    print("  Turbo solo :", r.Turbo)
    print("  Final      :", r[FINAL])
    display(Audio(destino))
demo[["Sample_ID", "Speaker_ID", "Disease", "Intelligibility", "Transcription", "Turbo", FINAL, "WER_turbo",
      "WER_final"]].to_csv(os.path.join(DEMO_DIR, "demo_turbo.csv"), index=False)
print("\nCopiados a", DEMO_DIR)

## 4. Pipeline en vivo
`transcribir_demo(ruta, perfil="")`: Groq transcribe con whisper-large-v3-turbo, se aplican las reglas y, si
corresponde, el LLM. El **perfil** es opcional: lo que la app sabe del usuario (nombres de su familia, temas). Va al
LLM con la instrucción de usarlo solo para resolver palabras dudosas.

In [ ]:
PREFIJO_PERFIL = ("Información sobre esta persona. Usala SOLO para resolver palabras dudosas (nombres propios, "
                  "lugares, temas recurrentes). No agregues nada que no esté respaldado por las hipótesis.\n"
                  "<contexto>\n{ctx}\n</contexto>\n\n")

def transcribir_api(ruta):
    from openai import OpenAI
    if "groq_asr" not in _cliente:
        _cliente["groq_asr"] = OpenAI(api_key=obtener_secreto("GROQ_API_KEY"), base_url=CFG["asr_base_url"],
                                      max_retries=3)
    with open(ruta, "rb") as f:
        r = _cliente["groq_asr"].audio.transcriptions.create(file=f, model=CFG["asr_modelo"], language="es",
                                                              temperature=0.0)
    return " ".join(r.text.split())

def corregir_texto(texto, perfil=""):
    hyps = hipotesis_para_llm(texto, [])
    base = hyps[0]
    if not USAR_LLM or not base.strip():
        return base
    contenido = (PREFIJO_PERFIL.format(ctx=perfil.strip()) if perfil and perfil.strip() else "") + formatear_hipotesis(hyps)
    try:
        salida = llamar_llm(EJEMPLOS + [{"role": "user", "content": contenido}])
    except Exception as e:
        print("Error de LLM:", type(e).__name__, str(e)[:200])
        return base
    salida = (salida or "").strip().strip('"“”«»').strip()
    nb, ns = len(base.split()), len(salida.split())
    return base if (not salida or ns < 0.5 * nb or ns > 1.5 * nb + 3) else salida

def transcribir_demo(ruta, perfil=""):
    t0 = time.time()
    crudo = transcribir_api(ruta)
    final = corregir_texto(crudo, perfil)
    print(f"Whisper grande (sin corregir): {crudo}\nNuestro sistema: {final}\n({time.time()-t0:.1f} s)")
    return crudo, final

if len(demo):
    transcribir_demo(os.path.join(DEMO_DIR, f"{demo.iloc[0].Sample_ID}.wav"))

## 5. App de Gradio
Micrófono o archivo, con un perfil opcional. `share=True` da un link público (72 h) que se puede abrir desde el
celular. Si Colab se desconecta, *Ejecutar todas* la levanta de nuevo en un par de minutos (todo sale de la caché).

In [ ]:
import gradio as gr

def _app(ruta, perfil):
    if not ruta:
        return "", ""
    return transcribir_demo(ruta, perfil or "")

with gr.Blocks(title="Whisper adaptado a habla con disartria") as app:
    gr.Markdown("### Reconocimiento de habla para personas con ELA, Parkinson o ACV\n"
                "Grabá o subí un audio en español. A la izquierda, Whisper grande tal cual; a la derecha, "
                "nuestro sistema (reglas anti-alucinación" + (" + corrección con LLM" if USAR_LLM else "") + ").")
    with gr.Row():
        entrada = gr.Audio(sources=["microphone", "upload"], type="filepath", label="Audio")
        perfil = gr.Textbox(label="Perfil del usuario (opcional)", lines=4,
                            placeholder="Ej.: Se llama Marta, tiene ELA. Su hija es Lucía y su marido Andrés.")
    with gr.Row():
        salida_w = gr.Textbox(label="Whisper grande (sin corregir)", lines=6)
        salida_n = gr.Textbox(label="Nuestro sistema", lines=6)
    gr.Button("Transcribir", variant="primary").click(_app, [entrada, perfil], [salida_w, salida_n])
    ejemplos = sorted(glob.glob(os.path.join(DEMO_DIR, "*.wav")))
    if ejemplos:
        gr.Examples([[e, ""] for e in ejemplos], [entrada, perfil])
app.launch(share=True, debug=False)